In [1]:
import pandas as pd
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn import datasets

In [2]:
df = datasets.load_diabetes(as_frame = True).frame

In [3]:
df.head(5)

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB


In [5]:
df.isnull().sum()

age       0
sex       0
bmi       0
bp        0
s1        0
s2        0
s3        0
s4        0
s5        0
s6        0
target    0
dtype: int64

In [6]:
x = df.drop("target", axis = 1)
y = df["target"]

In [7]:
x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state = 40
)

In [8]:
ss = StandardScaler()
y_scaled_train = ss.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_scaled_test = ss.transform(y_test.values.reshape(-1,1)).ravel()

In [9]:
model = SVR()
model.fit(x_train,y_scaled_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [10]:
y_pred = model.predict(x_test)
y_train_pred_scaled = model.predict(x_train)

In [11]:
print("r2 score =", r2_score(y_pred,y_scaled_test))
print("train r2: ", r2_score(y_scaled_train, y_train_pred_scaled))

r2 score = -0.4203098196795241
train r2:  0.7122906660929993


In [12]:
# Linear
model = SVR(kernel="linear")

model.fit(x_train, y_scaled_train)

y_test_pred_scaled = model.predict(x_test)
y_train_pred_scaled = model.predict(x_train)

print("train r2: ", r2_score(y_scaled_train, y_train_pred_scaled))
print("test r2: ", r2_score(y_scaled_test, y_test_pred_scaled))

train r2:  0.473849013468861
test r2:  0.3422248511006337


In [13]:
# poly
model = SVR(kernel="poly")

model.fit(x_train, y_scaled_train)

y_test_pred_scaled = model.predict(x_test)
y_train_pred_scaled = model.predict(x_train)

print("train r2: ", r2_score(y_scaled_train, y_train_pred_scaled))
print("test r2: ", r2_score(y_scaled_test, y_test_pred_scaled))

train r2:  0.6047981556911441
test r2:  0.28821039623513145


In [14]:
# sigmoid
model = SVR(kernel="sigmoid")

model.fit(x_train, y_scaled_train)

y_test_pred_scaled = model.predict(x_test)
y_train_pred_scaled = model.predict(x_train)

print("train r2: ", r2_score(y_scaled_train, y_train_pred_scaled))
print("test r2: ", r2_score(y_scaled_test, y_test_pred_scaled))

train r2:  -23.76171222780514
test r2:  -15.334463230773334


# Hyperparameter tuning usign GridSearchCV

In [15]:
param_grid = {
    "C" : [1, 2, 5, 10, 50, 100],
    "kernel" : ["linear","poly","sigmoid"],
    "epsilon" : [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7]
}

In [16]:
svr = SVR()
grid = GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    scoring="r2",
    cv=5
)
grid.fit(x_train,y_scaled_train)
print("best parameter = ", grid.best_params_)

best parameter =  {'C': 100, 'epsilon': 0.7, 'kernel': 'linear'}


In [20]:
best_model = SVR(C = 100, epsilon= 0.7, kernel= 'linear')
best_model.fit(x_train,y_scaled_train)
y_test_pred_scaled = best_model.predict(x_test)
y_train_pred_scaled = best_model.predict(x_train)

print("train r2: ", r2_score(y_scaled_train, y_train_pred_scaled))
print("test r2: ", r2_score(y_scaled_test, y_test_pred_scaled))

train r2:  0.5512025929765245
test r2:  0.3562253705656413


In [22]:
from sklearn.svm import LinearSVR

model = LinearSVR(C=10, epsilon=0.1, max_iter=5000)

model.fit(x_train,y_scaled_train)

y_test_pred_scaled = model.predict(x_test)
y_train_pred_scaled = model.predict(x_train)

print("train r2: ", r2_score(y_scaled_train, y_train_pred_scaled))
print("test r2: ", r2_score(y_scaled_test, y_test_pred_scaled))

train r2:  0.549186694284171
test r2:  0.3579519757066588
